# 15 — Full Evaluation Suite on Colab (GPU)

Runs all evaluation scripts on a GPU runtime for rigorous numbers.

## Nested evaluation design

All models are evaluated on the **same fixed 5,000-comment subset** (seeded) for
the paper's main comparison table — every model sees the exact same comments,
ensuring a perfectly fair head-to-head.

Fast models (toxic-bert, tfidf-svm, distilbert-student) are additionally run on
the **full 63k test set** as confirmation. With n=5000 the 95% CI on F1 is ±0.01,
so the 5k numbers are already reliable; the full-test run just confirms there's
no sampling artifact.

## Experiments

1. **Fair comparison (all 6 models, same 5k)** — main table for the report
2. **Full-test confirmation (3 fast models, 63k)** — parenthetical validation
3. **Fairness** — all models (incl. D2, D1-ft) with perturbation-augmented probes
4. **Encoder–decoder agreement** — toxic-bert vs D1-ft (the better decoder)
5. **Robustness** — bidirectional: evasion (toxic) + FP induction (non-toxic), n=1000

## Colab steps
1. Runtime → Change runtime type → **GPU**
2. Upload `DeepProjectHate_colab.zip` (~250 MB) and run the unzip + download cells
3. Run all cells (~2.5 hours total)
4. Download `data/processed/*.json` and `report/figures/*.png` back to laptop

**Estimated runtimes (T4 GPU):**
- Fair comparison (all models, 5k): ~45 min
- Full-test confirmation (fast models, 63k): ~1 hour
- Fairness + perturbation probes: ~10 min
- Agreement (D1-ft): ~15 min
- Robustness bidirectional: ~20 min

## 0. Setup — unzip project and download models + data

Upload `DeepProjectHate_colab.zip` (created by `zip_for_colab.py`) then run
the cells below. This downloads HuggingFace models and Jigsaw data directly
on Colab (~2 min on Colab's network), avoiding a 1.4 GB upload.

In [ ]:
!pip install -q transformers datasets scikit-learn pandas pyarrow accelerate

import zipfile, os
from pathlib import Path

zip_path = Path("DeepProjectHate_colab.zip")
if not zip_path.exists():
    from google.colab import files
    print("Upload DeepProjectHate_colab.zip:")
    uploaded = files.upload()
    zip_path = Path(list(uploaded.keys())[0])

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(".")
print("Unzipped. Contents:")
!ls -la

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT / "DeepProjectHate" / "src").exists():
    ROOT = ROOT / "DeepProjectHate"
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("ROOT", ROOT.resolve())

# Download HuggingFace models (toxic-bert + distilgpt2) — ~1 min on Colab
print("\n--- Downloading toxic-bert from HuggingFace ---")
!python scripts/download_toxic_bert.py

print("\n--- Downloading distilgpt2 from HuggingFace ---")
!python scripts/download_distilgpt2.py

# Download Jigsaw data — ~30 sec on Colab
print("\n--- Downloading Jigsaw data ---")
!python scripts/download_jigsaw.py

# Verify everything is in place
from paths import LOCAL_MODEL_DIR, LOCAL_DISTILGPT2_DIR, LOCAL_DISTILBERT_STUDENT_DIR, LOCAL_DISTILGPT2_D1_FT_DIR, TRAIN_CSV
checks = {
    "toxic-bert": LOCAL_MODEL_DIR.exists(),
    "distilgpt2": LOCAL_DISTILGPT2_DIR.exists(),
    "distilbert-student": LOCAL_DISTILBERT_STUDENT_DIR.exists(),
    "distilgpt2-d1-ft": LOCAL_DISTILGPT2_D1_FT_DIR.exists(),
    "jigsaw data": TRAIN_CSV.exists(),
}
for name, ok in checks.items():
    print(f"  {'✓' if ok else '✗'} {name}")
assert all(checks.values()), f"Missing: {[k for k,v in checks.items() if not v]}"
print("\nAll models and data ready!")

In [ ]:
# (Setup was done in the cells above)

In [ ]:
import torch
from device import get_device

dev = get_device()
print(f"Device: {dev}")
if dev.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — runs will be very slow!")

In [ ]:
from splits import split_counts
print("Dataset split sizes:")
print(split_counts().to_string(index=False))

## 1. Fair comparison — ALL models on same 5k subset (main table)

Every model sees the exact same 5,000 comments (seeded). This is the primary
comparison table for the report.

In [ ]:
!python scripts/run_model_comparison.py \
    --models toxic-bert tfidf-svm distilbert-student distilgpt2-d1 distilgpt2-d1-ft distilgpt2-d2 \
    --eval-n 5000

In [ ]:
import json, shutil
from paths import PROCESSED_DIR

src = PROCESSED_DIR / "model_comparison.json"
dst = PROCESSED_DIR / "model_comparison_5k_fair.json"
shutil.copy2(src, dst)
print(f"Saved 5k fair-comparison snapshot to {dst.name}")

with open(src) as f:
    mc5k = json.load(f)
print(f"  models: {list(mc5k['models'].keys())}")
print(f"  eval_n: {mc5k['eval_n']}")

## 2. Full-test confirmation — fast models only (63k)

Fast models (encoder, SVM, student) run on the full 63,978 test set to confirm
the 5k numbers are not a sampling artifact. Results merge into the same JSON
(the 5k decoder results are preserved).

In [ ]:
!python scripts/run_model_comparison.py --models toxic-bert tfidf-svm distilbert-student

In [ ]:
import json
from paths import PROCESSED_DIR

fair_path = PROCESSED_DIR / "model_comparison_5k_fair.json"
full_path = PROCESSED_DIR / "model_comparison.json"

with open(fair_path) as f:
    mc5k = json.load(f)

print("=" * 90)
print("FAIR COMPARISON — all models on same 5k test subset (PRIMARY TABLE)")
print("=" * 90)
for key, block in mc5k["models"].items():
    t = block["splits"].get("test", {})
    print(f"{key:25s}  params={block['n_params']:>10,}  "
          f"F1={t.get('f1', 0):.3f}  PR-AUC={t.get('pr_auc', 0):.3f}  "
          f"ROC-AUC={t.get('roc_auc', 0):.3f}  {t.get('ms_per_comment', 0):.1f} ms/comment")

print()
with open(full_path) as f:
    mc_full = json.load(f)

print("=" * 90)
print("FULL-TEST CONFIRMATION — fast models on 63k")
print("=" * 90)
for key in ["toxic-bert", "tfidf-svm", "distilbert-student"]:
    if key in mc_full["models"]:
        block = mc_full["models"][key]
        t = block["splits"].get("test", {})
        print(f"{key:25s}  F1={t.get('f1', 0):.3f}  PR-AUC={t.get('pr_auc', 0):.3f}  "
              f"ROC-AUC={t.get('roc_auc', 0):.3f}  (n={mc_full['eval_n'].get('test', '?')})")

## 3. Fairness — all models + D2 + D1-ft + perturbation probes

In [ ]:
!python scripts/run_fairness.py \
    --models toxic-bert tfidf-svm distilbert-student distilgpt2-d2 distilgpt2-d1-ft \
    --perturb

In [ ]:
with open(PROCESSED_DIR / "fairness_results.json") as f:
    fr = json.load(f)

print("Fairness summary (unperturbed):")
for key, block in fr["models"].items():
    print(f"  {key:25s}  identity_FPR={block['overall_fpr']:.3f}  "
          f"control_FPR={block['control_fpr']:.3f}")
    if "perturbed" in block:
        print(f"    perturbed FPR by perturbation:")
        for pname, pdata in block["perturbed"].items():
            print(f"      {pname:12s}  identity={pdata['overall_fpr']:.3f}  "
                  f"control={pdata['control_fpr']:.3f}")

## 4. Encoder–decoder agreement (toxic-bert vs D1-ft)

In [ ]:
!python scripts/run_agreement.py --n 500

In [ ]:
with open(PROCESSED_DIR / "agreement_results.json") as f:
    ag = json.load(f)

print(f"n={ag['n']}  overall agreement={ag['overall_enc_dec_agreement']:.3f}")
print(f"Disagreements on toxic: {ag['n_disagree_toxic']}/{ag['n']}")
print()
print("vs gold:")
for row in ag["vs_gold"]:
    print(f"  {row['label']:15s}  encoder={row['encoder_vs_gold']:.3f}  "
          f"decoder={row['decoder_vs_gold']:.3f}")

## 5. Robustness — bidirectional (evasion + FP induction), n=1000

In [ ]:
!python scripts/run_robustness.py --models toxic-bert tfidf-svm distilbert-student --n 1000

In [ ]:
with open(PROCESSED_DIR / "robustness_results.json") as f:
    rb = json.load(f)

print(f"Toxic probe: {rb['n_toxic']} comments  |  Non-toxic probe: {rb['n_nontoxic']} comments")
print()
for key, block in rb["models"].items():
    tp = block["toxic_probe"]
    np_block = block["nontoxic_probe"]
    print(f"=== {key} ===")
    print(f"  Toxic probe: baseline detection={tp['baseline_detection_rate']:.3f}")
    for pname, pd_ in tp["perturbations"].items():
        print(f"    {pname:12s}  evasion={pd_['evasion_rate']:.3f}")
    print(f"  Non-toxic probe: baseline FP rate={np_block['baseline_fp_rate']:.3f}")
    for pname, pd_ in np_block["perturbations"].items():
        print(f"    {pname:12s}  FP-induction={pd_['fp_induction_rate']:.3f}")
    print()

## 6. Summary — download these files

Copy back to your laptop:
- `data/processed/model_comparison_5k_fair.json` — **primary**: all 6 models on same 5k
- `data/processed/model_comparison.json` — merged: fast models at 63k, decoders at 5k
- `data/processed/fairness_results.json`
- `data/processed/agreement_results.json`
- `data/processed/robustness_results.json`
- `report/figures/*.png`

In [ ]:
import shutil
from pathlib import Path

out = Path("colab_results")
out.mkdir(exist_ok=True)

for f in (PROCESSED_DIR).glob("*.json"):
    shutil.copy2(f, out / f.name)
    print(f"Copied {f.name}")

fig_dir = Path("report/figures")
if fig_dir.exists():
    for f in fig_dir.glob("*.png"):
        shutil.copy2(f, out / f.name)
        print(f"Copied {f.name}")

print(f"\nAll results in {out.resolve()} — download this folder.")